# Day 3: Tokenizers

> This notebook documents **Day 3 only**.  
> Later days have separate notebooks.

## Overview

This Colab session explores the world of Tokenizers. You can run this notebook on a free CPU, or locally on your box if you prefer.

## Learning Objectives

- Understand the tokenization process (text → tokens → token IDs)
- Compare different tokenizers and their behaviors
- Learn about vocabulary and token counts
- Understand Instruct variants and chat templates
- Discover the crucial "Aha" moment about LLM inputs
- Explore multiple models: Llama 3.1, Phi-4, DeepSeek, QwenCoder

## Resources

- [Tokenizers Colab](https://colab.research.google.com/drive/1WD6Y2N7ctQi1X9wa6rpkg8UfyA4iSVuz?usp=sharing)
- [HuggingFace Tokenizers Docs](https://huggingface.co/docs/tokenizers)


## Colab Pro-Tips

### Pro-Tip 1: Warnings
Don't worry about warnings and messages!

### Pro-Tip 2: Misleading CUDA Errors
In the middle of running a Colab, you might get an error like:
```
Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]
```

**This is a super-misleading error message!** Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. `Kernel menu → Disconnect and delete runtime`
2. Reload the colab from fresh and `Edit menu → Clear All Outputs`
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!


## Setup

### 1. Sign in to Hugging Face

If you haven't already done so, create a free HuggingFace account at https://huggingface.co and navigate to Settings, then Create a new API token, giving yourself write permissions.

**IMPORTANT:** when you create your HuggingFace API key, please be sure to select read/write permissions for your key by clicking on the WRITE tab, otherwise you may get problems later.

Press the "key" icon on the side panel to the left, and add a new secret: `HF_TOKEN = your_token`


In [ ]:
# Log in to Hugging Face

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)


### 2. Check Google Colab GPU

> Note: This notebook can run on CPU, but GPU verification is still useful.


In [ ]:
# Check Google Colab GPU

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")


### 3. Imports


In [ ]:
from transformers import AutoTokenizer


## Accessing Llama 3.1 from Meta

In order to use the fantastic Llama 3.1, Meta does require you to sign their terms of service.

**Steps:**
1. Visit their model instructions page in Hugging Face: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B
2. At the top of the page are instructions on how to agree to their terms
3. If possible, you should use the same email as your huggingface account
4. In my experience approval comes in a couple of minutes
5. Once you've been approved for any 3.1 model, it applies to the whole 3.1 family of models

**Important Note:** Llama 3.1 is commonly used in industry, which is why it's included in this course.

**Troubleshooting:**
If the next cell gives you an error, then please check:
- Are you logged in to HuggingFace? Try running `login()` to check your key works
- Did you set up your API key with full read and write permissions?
- If you visit the Llama3.1 page with the link above, does it show that you have access to the model near the top?

For whatever reason, occasionally Meta doesn't approve access. If that happens to you, please follow the troubleshooting steps in the Colab.


## Basic Tokenization

### Loading Llama 3.1 Tokenizer


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)

text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
tokens


### Character, Word, and Token Count Comparison


In [ ]:
character_count = len(text)
word_count = len(text.split(' '))
token_count = len(tokens)
print(f"There are {character_count} characters, {word_count} words and {token_count} tokens")


### Decoding Tokens


In [ ]:
tokenizer.decode(tokens)


In [ ]:
tokenizer.batch_decode(tokens)


### Vocabulary


In [ ]:
# tokenizer.vocab
tokenizer.get_added_vocab()

len(tokenizer.vocab)


## Instruct Variants of Models

Many models have a variant that has been trained for use in Chats. These are typically labelled with the word "Instruct" at the end. They have been trained to expect prompts with a particular format that includes system, user and assistant prompts.

There is a utility method `apply_chat_template` that will convert from the messages list format we are familiar with, into the right input prompt for this model.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)


## Crucial "Aha" Moment

For 2.5 weeks, I've given you the impression that LLMs could receive a list of python dictionaries in some way:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
]
```

**But an LLM is just a Data Science model that takes a sequence of numbers and predicts the probability of the next number! You can't pass a bunch of Python objects into a statistical model!**

**And now you have the missing piece of the puzzle:**

1. The messages in OpenAI format get converted:
   - ...into a sequence of words with special tags to separate the System, User, Assistant prompt
2. Then the words are broken down into fragments - "tokens"
3. Then the tokens are replaced with Token IDs - and this is the input sequence

**The input to an LLM is a sequence of Token IDs. The output is the probability distribution of the next Token ID to follow this input.**

**That's it!**


## Trying New Models

We will now work with 3 models:
- **Phi-4** from Microsoft
- **DeepSeek 3.1** from DeepSeek AI
- **QwenCoder 2.5** from Alibaba Cloud


In [ ]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print("Llama:")
tokens = tokenizer.encode(text)
print(tokens)
print(tokenizer.batch_decode(tokens))
print("\nPhi 4:")
tokens = phi4_tokenizer.encode(text)
print(tokens)
print(phi4_tokenizer.batch_decode(tokens))


### Comparing Chat Templates Across Models


In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi 4:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))


### DeepSeek Tokenizer


In [ ]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print(tokenizer.encode(text))
print()
print(phi4_tokenizer.encode(text))
print()
print(deepseek_tokenizer.encode(text))


### Comparing All Chat Templates


In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nDeepSeek:")
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))


### QwenCoder Tokenizer (Code-Specific)


In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)
code = """
def hello_world(person):
  print("Hello", person)
"""
tokens = qwen_tokenizer.encode(code)
for token in tokens:
  print(f"{token}={qwen_tokenizer.decode(token)}")


## Key Learnings

### Tokenization Process
- Text → Tokens → Token IDs
- Different tokenizers produce different tokenizations for the same text
- Character count ≠ Word count ≠ Token count
- Tokens are fragments of words, not always whole words

### Vocabulary
- Each tokenizer has a vocabulary (mapping of tokens to IDs)
- Vocabulary size varies by model
- Special tokens are added to vocabulary

### Decoding
- `decode()` converts token IDs back to text
- `batch_decode()` can decode multiple sequences
- Round-trip: text → tokens → token IDs → text (should match original)

### Instruct Variants and Chat Templates
- Instruct models are trained for chat/conversation
- `apply_chat_template()` converts messages format to model-specific prompts
- Different models have different chat template formats
- Chat templates add special tokens for system/user/assistant roles

### The Crucial "Aha" Moment
- **LLMs take Token IDs as input, not Python objects**
- Messages format (list of dicts) is converted to token IDs
- Process: Messages → Text with tags → Tokens → Token IDs
- Output is probability distribution of next Token ID
- This is the missing piece connecting high-level APIs to model internals

### Multiple Models Comparison
- **Llama 3.1:** Meta's model
- **Phi-4:** Microsoft's model
- **DeepSeek 3.1:** DeepSeek AI model
- **QwenCoder 2.5:** Alibaba Cloud's code-specific model
- Each has different tokenization behavior and chat templates

### Practical Insights
- Can run tokenizers on CPU (no GPU needed)
- Tokenization is deterministic (same text → same tokens)
- Understanding tokenization helps debug model behavior
- Token limits are based on token count, not character/word count
